# RAG 시스템 고급 기법 

- 다양한 도구와 최적화 전략 
    - 다양한 Document Loader 탐색 
    - TextSplitter 비교 및 선택 
    - Retriever 최적화 기법 

# 1. API Key 로드 

In [1]:
from dotenv import load_dotenv 

load_dotenv()

True

# 2. Document Loader

- https://docs.langchain.com/oss/python/integrations/document_loaders#all-document-loaders

## 2.1 CSVLoader

- 문서(Document객체)당 한 행씩 로드 됨
- 한 행을 임베딩하게되면 LLM이 잘 인지하지 못하는 경우도 많음 

In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader 

docs = CSVLoader("../data/titanic/train.csv").load()

print(type(docs))
print(len(docs))
docs[0]

<class 'list'>
891


Document(metadata={'source': '../data/titanic/train.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S')

In [3]:
print(docs[0].page_content)

PassengerId: 1
Survived: 0
Pclass: 3
Name: Braund, Mr. Owen Harris
Sex: male
Age: 22
SibSp: 1
Parch: 0
Ticket: A/5 21171
Fare: 7.25
Cabin: 
Embarked: S


In [4]:
print(docs[0].metadata)

{'source': '../data/titanic/train.csv', 'row': 0}


## 2.3 JSONLoader 

### 2.3.1 json 라이브러리로 로드 

In [7]:
import json 

file_path = "../data/musinsa_origin.json"
with open(file_path, "r", encoding="utf-8") as f:
    json_data = json.load(f) # json파일 로드 

print("페이지 수:", len(json_data))
print("페이지당 상품 수:", len(json_data[0]["data"]["list"]))
print(json_data[0])

페이지 수: 49
페이지당 상품 수: 30
{'data': {'list': [{'goodsNo': 5923652, 'goodsName': '언이븐 골지 슬림 버튼 롱슬리브 화이트', 'goodsLinkUrl': 'https://www.musinsa.com/products/5923652', 'thumbnail': 'https://image.msscdn.net/images/goods_img/20260122/5923652/5923652_17701632436291_500.jpg', 'displayGenderText': '여성', 'isSoldOut': False, 'normalPrice': 46000, 'price': 33900, 'couponPrice': 30510, 'saleRate': 26, 'couponSaleRate': 34, 'hasOptionPrice': False, 'brand': 'chicks', 'brandName': '칙스', 'brandLinkUrl': 'https://www.musinsa.com/brand/chicks', 'reviewCount': 0, 'reviewScore': 0, 'isOptionVisible': True, 'isAd': False, 'adTooltip': None, 'plusDeliveryGuideText': '', 'infoLabelList': [], 'imageLabelList': [], 'clickTrackers': [], 'impressionTrackers': [], 'snap': None, 'score': 0.0, 'isPlusDelivery': False, 'usedConditionGrade': None}, {'goodsNo': 5886659, 'goodsName': 'MIL SERIES SWEAT(152nd)_BROWN', 'goodsLinkUrl': 'https://www.musinsa.com/products/5886659', 'thumbnail': 'https://image.msscdn.net/images

In [8]:
# 제품 정보만 추출 
docs = []
for data in json_data:
    docs.extend(data["data"]["list"])

print(len(docs))
print(docs[0])

1470
{'goodsNo': 5923652, 'goodsName': '언이븐 골지 슬림 버튼 롱슬리브 화이트', 'goodsLinkUrl': 'https://www.musinsa.com/products/5923652', 'thumbnail': 'https://image.msscdn.net/images/goods_img/20260122/5923652/5923652_17701632436291_500.jpg', 'displayGenderText': '여성', 'isSoldOut': False, 'normalPrice': 46000, 'price': 33900, 'couponPrice': 30510, 'saleRate': 26, 'couponSaleRate': 34, 'hasOptionPrice': False, 'brand': 'chicks', 'brandName': '칙스', 'brandLinkUrl': 'https://www.musinsa.com/brand/chicks', 'reviewCount': 0, 'reviewScore': 0, 'isOptionVisible': True, 'isAd': False, 'adTooltip': None, 'plusDeliveryGuideText': '', 'infoLabelList': [], 'imageLabelList': [], 'clickTrackers': [], 'impressionTrackers': [], 'snap': None, 'score': 0.0, 'isPlusDelivery': False, 'usedConditionGrade': None}


In [ ]:
docs 

# 해당 데이터를 추가로 더 분할을 진행할 필요가 있는지 판단하기 
# 현재 json데이터는 제품에 대한 모든 정보가 하나의 dict에 담겨서 구분돼있기 때문에 
# 추가로 분할을 진행할 필요가 없음 
# 다만, 한 개의 dict의 내용이 너무 길 경우에는 embedding시 모든 정보를 수치화하지 못할 수도 있기 때문에 
# 그럴 경우에만 추가로 분할을 진행할 필요가 있음  

[{'goodsNo': 5923652,
  'goodsName': '언이븐 골지 슬림 버튼 롱슬리브 화이트',
  'goodsLinkUrl': 'https://www.musinsa.com/products/5923652',
  'thumbnail': 'https://image.msscdn.net/images/goods_img/20260122/5923652/5923652_17701632436291_500.jpg',
  'displayGenderText': '여성',
  'isSoldOut': False,
  'normalPrice': 46000,
  'price': 33900,
  'couponPrice': 30510,
  'saleRate': 26,
  'couponSaleRate': 34,
  'hasOptionPrice': False,
  'brand': 'chicks',
  'brandName': '칙스',
  'brandLinkUrl': 'https://www.musinsa.com/brand/chicks',
  'reviewCount': 0,
  'reviewScore': 0,
  'isOptionVisible': True,
  'isAd': False,
  'adTooltip': None,
  'plusDeliveryGuideText': '',
  'infoLabelList': [],
  'imageLabelList': [],
  'clickTrackers': [],
  'impressionTrackers': [],
  'snap': None,
  'score': 0.0,
  'isPlusDelivery': False,
  'usedConditionGrade': None},
 {'goodsNo': 5886659,
  'goodsName': 'MIL SERIES SWEAT(152nd)_BROWN',
  'goodsLinkUrl': 'https://www.musinsa.com/products/5886659',
  'thumbnail': 'https://im

In [ ]:
# vectorstore에 전달할 수 있는 자료형으로 변환 필요
# docs를 document 객체로 변환 진행 
from langchain_core.documents import Document 

docs = []

for page in json_data:
    for item in page["data"]["list"]:
        docs.append(
            Document(
                page_content=str(item),
                metadata={
                    "상품명": item.get("goodsName"),
                    "브랜드": item.get("brandName"),
                    "가격": item.get("price"),
                }
            )
        )

print(len(docs))
docs[0]

1470


Document(metadata={'상품명': '언이븐 골지 슬림 버튼 롱슬리브 화이트', '브랜드': '칙스', '가격': 33900}, page_content="{'goodsNo': 5923652, 'goodsName': '언이븐 골지 슬림 버튼 롱슬리브 화이트', 'goodsLinkUrl': 'https://www.musinsa.com/products/5923652', 'thumbnail': 'https://image.msscdn.net/images/goods_img/20260122/5923652/5923652_17701632436291_500.jpg', 'displayGenderText': '여성', 'isSoldOut': False, 'normalPrice': 46000, 'price': 33900, 'couponPrice': 30510, 'saleRate': 26, 'couponSaleRate': 34, 'hasOptionPrice': False, 'brand': 'chicks', 'brandName': '칙스', 'brandLinkUrl': 'https://www.musinsa.com/brand/chicks', 'reviewCount': 0, 'reviewScore': 0, 'isOptionVisible': True, 'isAd': False, 'adTooltip': None, 'plusDeliveryGuideText': '', 'infoLabelList': [], 'imageLabelList': [], 'clickTrackers': [], 'impressionTrackers': [], 'snap': None, 'score': 0.0, 'isPlusDelivery': False, 'usedConditionGrade': None}")

### 2.2.2 JSONLoader

- LangChain 프레임워크 활용
- 단점: 데이터 디코딩이 안 되므로 결과를 확인할 수가 없음 (모델 학습에는 문제되지 않음)

In [ ]:
# !pip install jq

  Using cached jq-1.11.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (7.0 kB)
Using cached jq-1.11.0-cp311-cp311-macosx_11_0_arm64.whl (422 kB)


In [5]:
from langchain_community.document_loaders import JSONLoader 

file_path = "../data/musinsa_origin.json"
loader = JSONLoader(
    file_path=file_path,
    jq_schema=".[].data.list.[].goodsName",  # 어디를 가져올지 지정하는 필터로, ".[]" -> 배열 내의 객체를 푸는 것을 의미 (리스트 안의 객체 하나씩 가져옴을 의미)
    text_content=True, # jq_schema의 결과가 문자열인지 아닌지에 대한 여부
)

docs = loader.load()

In [6]:
docs

[Document(metadata={'source': '/Users/yoon/Desktop/sesac/data/musinsa_origin.json', 'seq_num': 1}, page_content='언이븐 골지 슬림 버튼 롱슬리브 화이트'),
 Document(metadata={'source': '/Users/yoon/Desktop/sesac/data/musinsa_origin.json', 'seq_num': 2}, page_content='MIL SERIES SWEAT(152nd)_BROWN'),
 Document(metadata={'source': '/Users/yoon/Desktop/sesac/data/musinsa_origin.json', 'seq_num': 3}, page_content='투톤 슬리브리스 레이어드 롱슬리브 [CHARCOAL]'),
 Document(metadata={'source': '/Users/yoon/Desktop/sesac/data/musinsa_origin.json', 'seq_num': 4}, page_content='VTG 워싱 바라클라바 후드 티셔츠_피그먼트 차콜'),
 Document(metadata={'source': '/Users/yoon/Desktop/sesac/data/musinsa_origin.json', 'seq_num': 5}, page_content='이엘 레이어드 카라 긴팔 니트'),
 Document(metadata={'source': '/Users/yoon/Desktop/sesac/data/musinsa_origin.json', 'seq_num': 6}, page_content='COMFORT WOOL HENLEY NECK KNIT [BLACK]'),
 Document(metadata={'source': '/Users/yoon/Desktop/sesac/data/musinsa_origin.json', 'seq_num': 7}, page_content='ASYMMETRIC LOGO KNIT PULLO

In [7]:
print(len(docs))
docs[0].page_content

1470


'언이븐 골지 슬림 버튼 롱슬리브 화이트'

# 3. 문서 분할 

## 3.1 CharacterTextSplitter

- `\n\n` 기준으로 텍스트를 분할 (즉, 문단을 기준으로 분할됨)
    - 문장을 기준으로 분할하고 싶을 경우: `\n`사용
- 문자 수로 청크 크기 결정 
- 즉, 
    - 지정된 chunk_size 이내에서 `\n\n`를 찾은 후 `\n\n`를 발견하면 `\n\n`를 기준으로 자름  
    - 예를 들어 chunk_size가 100인데 93글자 쯤에서 `\n\n`를 발견하면 그 부분을 기준으로 chunk를 자름 
    - 근데 만약 chunk_size를 넘겨서 `\n\n`를 발견하여 자를 경우 자르긴 자르지만 경고문자가 뜸 

In [10]:
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import CharacterTextSplitter

# 1. 문서 로드
docs = PyMuPDFLoader("../data/하나은행_전세자금대출_상품설명서.pdf").load()
print("docs 개수:", len(docs))

# 2. splitter 정의 및 문서 분할 
splitter = CharacterTextSplitter(
    separator="\n\n", # 분할 기준 
    chunk_size=600,
    chunk_overlap=100
)
docs_split = splitter.split_documents(docs)

print(type(docs_split))
print(len(docs_split))
docs_split[0]

docs 개수: 26
<class 'list'>
26


Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-12-27T10:49:01+09:00', 'source': '../data/하나은행_전세자금대출_상품설명서.pdf', 'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-12-27T10:49:05+09:00', 'trapped': '', 'modDate': "D:20241227104905+09'00'", 'creationDate': "D:20241227104901+09'00'", 'page': 0}, page_content='5-06-0900(26-1) (2025.01 개정)\n(보존년한 : 완제일로부터 10년)\n준법감시인 심의필 제 2024-설명서-280 호\n(은행보관용)\t\n전세자금대출 상품설명서\n- 고객님께서는 상품 가입 전 아래 사항을 반드시 확인 · 숙지하여 주시기 바랍니다 -\n본인확인\n담당\n책임자\n지점장\n◈ \x07이 설명서는 금융소비자의 권익 보호 및 대출상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및 관련 규정에 \n의거, 은행의 내부 통제절차를 거쳐 대출상품의 주요 내용을 쉽게 이해할 수 있도록 작성한 자료입니다.\n◈ \x07설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우, 추후 해당 \n내용과 관련한 권리구제가 어려울 수 있습니다.\n유사 상품과 구별되는 특징\n¶\x07 \x07전세자금대출은 세입자(임차인)가 임대인으로부터 전세금(임차금)을 돌려받을 권리(임차보증금반환채권)를 담보로 하여\n(질권설정

## 3.2 RecursiveCharacterTextSplitter

- 문자 분할 목록을 매개변수로 받아 동작(텍스트를 분할) 
    - 기본 문자 분할 목록: `["\n\n", "\n", " ", ""]`
    - 단락 -> 문장 -> 단어 순으로 재귀적으로 분할
    - 이는 해당 단위가 의미적으로 가장 강하게 연관된 텍스트 조각으로 간주되기 때문
    

In [11]:
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 문서 로드
docs = PyMuPDFLoader("../data/하나은행_전세자금대출_상품설명서.pdf").load()
print("docs 개수:", len(docs))

# 2. splitter 정의 및 문서 분할 
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # 분할 기준 
    chunk_size=600,
    chunk_overlap=100
)
docs_split = splitter.split_documents(docs)

print(type(docs_split))
print(len(docs_split))
docs_split[0]

docs 개수: 26
<class 'list'>
92


Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-12-27T10:49:01+09:00', 'source': '../data/하나은행_전세자금대출_상품설명서.pdf', 'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-12-27T10:49:05+09:00', 'trapped': '', 'modDate': "D:20241227104905+09'00'", 'creationDate': "D:20241227104901+09'00'", 'page': 0}, page_content='5-06-0900(26-1) (2025.01 개정)\n(보존년한 : 완제일로부터 10년)\n준법감시인 심의필 제 2024-설명서-280 호\n(은행보관용)\t\n전세자금대출 상품설명서\n- 고객님께서는 상품 가입 전 아래 사항을 반드시 확인 · 숙지하여 주시기 바랍니다 -\n본인확인\n담당\n책임자\n지점장\n◈ \x07이 설명서는 금융소비자의 권익 보호 및 대출상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및 관련 규정에 \n의거, 은행의 내부 통제절차를 거쳐 대출상품의 주요 내용을 쉽게 이해할 수 있도록 작성한 자료입니다.\n◈ \x07설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우, 추후 해당 \n내용과 관련한 권리구제가 어려울 수 있습니다.\n유사 상품과 구별되는 특징\n¶\x07 \x07전세자금대출은 세입자(임차인)가 임대인으로부터 전세금(임차금)을 돌려받을 권리(임차보증금반환채권)를 담보로 하여\n(질권설정

# 4. Embedding

- 차원이 높을 수록 정교한 표현 가능 (절대적이지 않고 문서마다 적절하게 잘 표현되는 차원을 찾아야함)
- 어떤 임베딩 알고리즘을 쓰느냐에 따라 사용자 입력값과 유사도를 계산했을 때 유사도가 다르게 나올 수 있음 
- OpenAI 임베딩 모델
    - 다국어 지원이 되는 모델 중 가장 성능이 무난해서 많이 쓰임 
    - 좋은 모델 같은 경우는 모델 사이즈가 커서 GPU환경에서 돌려야할 수도 있음, 
        - OpenAI의 임베딩 모델을 쓰는 경우엔 OpenAI의 자원을 빌려쓰기 때문에 무거운 모델도 일반 노트북에서 돌릴 수 있음 
    - 하지만 비용 발생 
    - https://platform.openai.com/docs/guides/embeddings#embedding-models
- HuggingFace
    - 한국어 처리 성능이 좋은 오픈 소스 모델을 쓰면 됨 (무료)
    - 대신 일반 노트북에 GPU가 없기 때문에 임베딩 하는 데 시간이 오래걸릴 수 있음 
- splitter를 통해 문서가 단락화 되면 한 단락을 임베딩 모델이 지원하는 차원으로 임베딩처리 됨 
    - 즉, 어떤 모델이 임베딩하는 차원이 1024차원이면 한 단락이 1024차원으로 임베딩 됨

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# 1. 문서 로드
docs = PyMuPDFLoader("../data/하나은행_전세자금대출_상품설명서.pdf").load()
print("docs 개수:", len(docs))

# 2. splitter 정의 및 문서 분할 
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # 분할 기준 
    chunk_size=600,
    chunk_overlap=100
)
docs_split = splitter.split_documents(docs)

# 3. embedding 객체 정의 
embedding = OpenAIEmbeddings(model="text-embedding-ada-002")
embedded_docs = embedding.embed_documents([page.page_content for page in docs_split])

print("embedded_docs 자료형:", type(embedded_docs), "\n")
print("첫 번째 요소 자료형:", type(embedded_docs[0]), "\n")
print("embedded_dcos 요소 개수:", len(embedded_docs), "\n") # 임베딩된 청크 개수 
print("embedded_docs 요소별 차원 수:", len(embedded_docs[0]), "\n") # 총 사전 개수 (임베딩 차원)
print("첫 번째 벡터값 확인:\n", embedded_docs[0][:10])

docs 개수: 26
embedded_docs 자료형: <class 'list'> 

첫 번째 요소 자료형: <class 'list'> 

embedded_dcos 요소 개수: 92 

embedded_docs 요소별 차원 수: 1536 

첫 번째 벡터값 확인:
 [-0.008472992107272148, -0.011148311197757721, 0.012778261676430702, -0.05155320093035698, -0.031306054443120956, 0.013761733658611774, -0.014552637934684753, -0.02112746052443981, -0.04332779720425606, -0.007021854165941477]


# 5. Reranker 

- reranking을 지원하는 cohere라이브러리 (cohere사에서 지원)
- 한 달에 1000건으로 API사용 제한이 있음 
- Cohere의 API를 활용해 최소한의 코드로 구현할 수 있음
- 한국어 성능도 나쁘지 않다고 평가 됨
- 홈페이지: https://dashboard.cohere.com/api-keys
    - cohere 홈페이지의 dashboard에서 API Key 발급 (COHERE_API_KEY환경변수 이름으로 저장)
- **CohereRerank**클래스
    - langchain과 cohere를 연동해 reranking을 지원하는 클래스 

In [ ]:
# !pip install -qU cohere langchain_cohere

In [ ]:
# !pip install langchain_classic

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_cohere import CohereRerank
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

# 1. 문서 로드
docs = PyMuPDFLoader("../data/하나은행_전세자금대출_상품설명서.pdf").load()
print("docs 개수:", len(docs))

# 2. splitter 정의 및 문서 분할 
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # 분할 기준 
    chunk_size=600,
    chunk_overlap=100
)
docs_split = splitter.split_documents(docs)

# 3. embedding 객체 정의 
embedding = OpenAIEmbeddings(model="text-embedding-ada-002")

# 4. vectorstore 정의
vectorstore = Chroma(
    embedding_function=embedding,
    persist_directory="./Chroma",
    collection_name="sesac1"
)

# 5. retriever, reranker 정의 
retriever = vectorstore.as_retriever()
reranker = CohereRerank(
    model="rerank-v3.5", # cohere에서 지원하는 reranking 모델
    top_n=3 # 최종적으로 가장 관련성이 높은 3개의 결과만 반환 
)

# 6. 압축기 정의
compression_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=reranker # 압축기 자리에 reranker 전달 
)

# 7. 검색 실행 
input_data = "중도상환 수수료가 얼마인가요?"
compressed_docs = compression_retriever.invoke(input_data)
compressed_docs # 가장 유사도가 높은 3개의 청크 반환

docs 개수: 26


/var/folders/np/0f8tqn_j13l5c_yx0hg3k67r0000gn/T/ipykernel_54869/4154841035.py:24: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


[Document(metadata={'file_path': '../data/하나은행_전세자금대출_상품설명서.pdf', 'title': '', 'modDate': "D:20241227104905+09'00'", 'moddate': '2024-12-27T10:49:05+09:00', 'format': 'PDF 1.4', 'keywords': '', 'author': '', 'creationDate': "D:20241227104901+09'00'", 'source': '../data/하나은행_전세자금대출_상품설명서.pdf', 'creationdate': '2024-12-27T10:49:01+09:00', 'page': 12, 'trapped': '', 'producer': 'Adobe PDF Library 17.0', 'subject': '', 'total_pages': 26, 'creator': 'Adobe InDesign 19.4 (Windows)', 'relevance_score': 0.040712103}, page_content='5-06-0900(26-13) (2025.01 개정)\n(보존년한 : 완제일로부터 10년)\n준법감시인 심의필 제 2024-설명서-280 호\n용    어\n설    명\n대위변제\n■ \x07채무자의 채무를 제3자가 대신 변제하는 행위로서 대신 변제해준 제3자(대위변제자)는 채무자에 대\n하여 구상권(기존 채권자를 대신하여 채무자에게 채무 상환을 청구할 수 있는 권리)을 취득합니다.\n채무인수\n■ \x07채무의 동일성을 그대로 유지하면서 그 채무를 기존 채무자(구채무자)로부터 제3자(신채무자)에게 \n이전하는 계약을 의미합니다.\n■ \x07일반적으로 담보대출을 받은 채무자가 담보물을 제3자(매수인)에게 매각하면서 해당 담보대\n출도 제3자(매수인)에게 이전하고자 할 때 활용됩니다.\n채권양도\n■ \x07채권의 동일성을 그대로 유지하면서 그 채권을 기존 채권자(구채권자)로부터 제3자(신채권자)\n에게 이전하는 계약을 의미합니다

In [3]:
for doc in compressed_docs:
    # print(doc.metadata["bank"])
    print("페이지:", doc.metadata["page"])
    print(doc.page_content)
    print("\n", "-"*70)

페이지: 12
5-06-0900(26-13) (2025.01 개정)
(보존년한 : 완제일로부터 10년)
준법감시인 심의필 제 2024-설명서-280 호
용    어
설    명
대위변제
■ 채무자의 채무를 제3자가 대신 변제하는 행위로서 대신 변제해준 제3자(대위변제자)는 채무자에 대
하여 구상권(기존 채권자를 대신하여 채무자에게 채무 상환을 청구할 수 있는 권리)을 취득합니다.
채무인수
■ 채무의 동일성을 그대로 유지하면서 그 채무를 기존 채무자(구채무자)로부터 제3자(신채무자)에게 
이전하는 계약을 의미합니다.
■ 일반적으로 담보대출을 받은 채무자가 담보물을 제3자(매수인)에게 매각하면서 해당 담보대
출도 제3자(매수인)에게 이전하고자 할 때 활용됩니다.
채권양도
■ 채권의 동일성을 그대로 유지하면서 그 채권을 기존 채권자(구채권자)로부터 제3자(신채권자)
에게 이전하는 계약을 의미합니다.
■ 임차인인 채무자가 임대인(구채권자)에게 갖는 임차보증금반환채권을 은행(신채권자)에 양도
(담보로 제공)하여 대출을 받는 형태가 대표적입니다.
기한의 이익(상실)
■ 계약의 내용에 기한이 존재함으로서 당사자가 받는 이익을 의미합니다.

 ----------------------------------------------------------------------
페이지: 25
5-06-0900(26-26) (2025.01 개정)
(보존년한 : 완제일로부터 10년)
준법감시인 심의필 제 2024-설명서-280 호
용    어
설    명
대위변제
■ 채무자의 채무를 제3자가 대신 변제하는 행위로서 대신 변제해준 제3자(대위변제자)는 채무자에 대
하여 구상권(기존 채권자를 대신하여 채무자에게 채무 상환을 청구할 수 있는 권리)을 취득합니다.
채무인수
■ 채무의 동일성을 그대로 유지하면서 그 채무를 기존 채무자(구채무자)로부터 제3자(신채무자)에게 
이전하는 계약을 의미합니다.
■ 일반적으로 담보대출을 받은 채무자가 담보물을 제3자(매수인)에게 매각하면서 해

# 6. 최종 프롬프트 작성 및 모델 응답 생성 

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI 

system_prompt = """\
당신은 대출 전문 상담사입니다. 아래 제공된 상품설명서 내용을 바탕으로 고객의 질문에 정확하게 답변하세요.

답변 원칙:
1. 어느 은행의 몇페이지 정보인지 언급하세요.
2. 문서에 있는 정보만 사용하세요.
3. 금리, 수수료 등 구체적 수치를 명시하세요.
4. 정보가 없으면 "해당 정보는 상품설명서에 없습니다. 1599-8000으로 문의하세요"라고 답하세요.
5. 전문 용어는 쉽게 풀어서 설명하세요.
6. 고객에게 불리한 조건(연체이자, 기한이익상실 등)은 반드시 안내하세요.\
"""

user_prompt = """
사용자 질의에 대해 아래의 context를 참고하여 답하세요.

--사용자 질의--
{question}

--context--
{context}
"""

final_template = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", user_prompt)
])

model = ChatOpenAI(model="gpt-5-nano", temperature=0)
chain = final_template | model | StrOutputParser()

input_data = "중도상환 수수료는 얼마인가요?"
response = chain.invoke({"question": input_data, "context": compressed_docs})
print(response)

은행/문서 출처: 하나은행 전세자금대출 상품설명서, 페이지 12페이지(참고), 페이지 16페이지, 페이지 25페이지 등에서 관련 내용이 수록되어 있습니다.

다만 중도상환 수수료의 구체적 금액은 상품설명서에 기재되어 있지 않습니다. 해당 정보는 상품설명서에 없습니다. 1599-8000으로 문의하세요
